# ML1 - GridSearch with DecisionTreeRegressor, RandomForestRegressor

This notebook follows the ML1_Pipeline_Model notebook and focuses on:
1. Using DecisionTreeRegressor and RandomForestRegressor (or others)  
2. Hyperparameter tuning with GridSearchCV  
3. Cross-validation

## 1. Importing Libraries

In [4]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import cross_val_score, train_test_split

## 2. Loading and Preparing Data with the Pipeline

We follow the same preparation steps as in the previous notebook.


In [13]:
# Load the data from the pickle file.
df = pd.read_pickle('arbres_clean_Laura.pkl')

categorical_variables = df.select_dtypes(
    include=['object', 'string', 'category']
).columns.tolist()

numerical_variables = df.select_dtypes(include = ['number']).columns.tolist()
print(categorical_variables,numerical_variables)

#pipeline
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
    ('encoder', OneHotEncoder())])

numerical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())])

preprocessor = ColumnTransformer([
    ('cat', categorical_pipeline, categorical_variables),
    ('num', numerical_pipeline, numerical_variables)])

print(type(X_transformed))
print(X_transformed.shape)

['sous_categorie', 'adr_secteur', 'genre_bota', 'espece', 'portarbre', 'structure'] ['stadededeveloppement', 'anneedeplantation', 'hauteurarbre', 'latitude', 'longitude']
<class 'scipy.sparse._csr.csr_matrix'>
(29526, 434)


In [ ]:
#récupération du df de travail, qu'on va appeler arbres
X_transformed = preprocessor.fit_transform(df)

arbres = pd.DataFrame.sparse.from_spmatrix(
    X_transformed,
    columns=preprocessor.get_feature_names_out(),
    index=df.index
)

arbres.sample(10)

,cat__sous_categorie_ESP065,cat__sous_categorie_ESP151,cat__sous_categorie_ESP174,cat__sous_categorie_ESP187,cat__adr_secteur_1,cat__adr_secteur_2,cat__adr_secteur_3,cat__adr_secteur_38151,cat__adr_secteur_38158,cat__adr_secteur_38249,...,cat__structure_CCAS,cat__structure_Ecole,cat__structure_Métropole de Grenoble,cat__structure_Unknown,cat__structure_Ville de Grenoble,num__stadededeveloppement,num__anneedeplantation,num__hauteurarbre,num__latitude,num__longitude
3445,1.0,0,0,0,0,0,1.0,0,0,0,...,0,0,0,0,1.0,0.399762,-0.849844,2.054104,-1.405581,-1.276485
4834,0,1.0,0,0,0,0,0,0,0,0,...,0,0,0,0,1.0,0.399762,-1.400079,0.391564,0.742261,1.178486
34,0,0,1.0,0,0,0,0,0,0,0,...,0,0,0,0,1.0,0.399762,1.406121,-1.270977,-1.231285,-0.117448
9357,0,1.0,0,0,1.0,0,0,0,0,0,...,0,0,1.0,0,0,0.399762,0.470721,0.391564,1.057233,-0.952038
6917,0,0,1.0,0,0,1.0,0,0,0,0,...,0,0,0,0,1.0,0.399762,-0.079514,-1.270977,1.004205,0.843705
29922,0,1.0,0,0,0,0,1.0,0,0,0,...,0,0,1.0,0,0,0.399762,1.406121,0.391564,-0.635363,-1.876328
8373,0,0,1.0,0,0,0,0,0,0,0,...,0,0,0,0,1.0,0.399762,0.30565,-1.270977,0.343719,1.613342
19338,0,0,1.0,0,0,0,0,0,0,0,...,0,0,0,0,1.0,0.399762,-0.849844,0.391564,-1.016647,0.893097
9667,0,0,1.0,0,0,0,0,0,0,0,...,0,0,1.0,0,0,0.399762,-0.849844,0.391564,-0.151787,1.621544
23523,0,0,1.0,0,0,1.0,0,0,0,0,...,0,0,0,0,1.0,0.399762,-1.124961,0.391564,1.482191,0.0473


In [17]:
arbres.columns

Index(['cat__sous_categorie_ESP065', 'cat__sous_categorie_ESP151',
       'cat__sous_categorie_ESP174', 'cat__sous_categorie_ESP187',
       'cat__adr_secteur_1', 'cat__adr_secteur_2', 'cat__adr_secteur_3',
       'cat__adr_secteur_38151', 'cat__adr_secteur_38158',
       'cat__adr_secteur_38249',
       ...
       'cat__structure_CCAS', 'cat__structure_Ecole',
       'cat__structure_Métropole de Grenoble', 'cat__structure_Unknown',
       'cat__structure_Ville de Grenoble', 'num__stadededeveloppement',
       'num__anneedeplantation', 'num__hauteurarbre', 'num__latitude',
       'num__longitude'],
      dtype='object', length=434)

## 3. Initial Cross-Validation

Before tuning hyperparameters, let's evaluate a model with default parameters using cross-validation.


In [16]:
# Initial cross-validation without hyperparameter tuning  
# We can also print the MSE or MAE scores from each cross-validation fold.

annee_de_plantation = arbres.pop('anneedeplantation')

lin_reg = LinearRegression()

X_train, X_test, y_train, y_test = train_test_split(arbres, annee_de_plantation, random_state = 28)

KeyError: 'anneedeplantation'

## 4. 🌳 Modeling with GridSearchCV

Let's choose two machine learning models.

Evaluate the models with hyperparameter tuning (testing different hyperparameter values).  
Optimize the models as you wish, for example for Decision Tree and Random Forest:

### 🔍 Hyperparameters Tested
- `max_depth`: maximum depth of the trees [3, 4, 5, 10]  
- `min_samples_leaf`: minimum number of samples required at a leaf [0.1, 0.05]  
- `n_estimators` (for Random Forest only): number of trees in the forest [50, 100, 200]

Let's use GridSearchCV to automate the search for the best hyperparameters.

In [ ]:
# Hyperparameter tuning with GridSearchCV, and we can print the MSE or MAE scores for each cross-validation fold.



## 5. Final Evaluation on the Test Set

In [ ]:
# Code for evaluating the best model on the test set.

## 6. Visualization of Predictions  
One model will suffice.

In [ ]:
# Visualisation of predictions.

## [OPTIONAL] 7.Feature Importance

Using the random forest, how can we determine which features are the most important?

In [ ]:
# Code. In your opinion, how is feature importance calculated in Random Forest?

Typically, one might consider using backward elimination on feature importance to select the best subset of features, but this can be time-consuming if there are many features and the model takes a while to train, as is the case here. Instead, we can use Lasso-regularized regression to eliminate features, before retraining a RandomForest on the selected features. Create a pipeline with `LassoCV` as the selector and a RandomForest as the model with the best hyperparameters found by the previous grid-search. Evaluate on the test set.

In [ ]:
# Code

## 8. [OPTIONAL]  Compare more different models

Compare more regression models with one another (for example, in addition to linear regression, decision trees, and forests, use `GradientBoostingRegressor`, and the novel `TabICLRegressor` if you have time ( https://github.com/soda-inria/tabicl ))), by performing appropriate hyperparameter tuning for each model:

In [1]:
# Code. Search which hyper-parameters are worth tuning for each model

Choose the model that performs best and its optimal hyperparameters, then select the best subset of features for this model, and finally perform one last grid search on your model’s hyperparameters using the selected features to obtain your final model. Evaluate it on the test set.

In [ ]:
# Code

## Conclusion